# 🗄️ Day 03a: Database Choices — SQL vs NoSQL

---

## 🎯 What You'll Master Today
- SQL vs NoSQL decision framework
- Database types and when to use each
- Read replicas & write masters
- Connection pooling & indexing strategy
- Database per service pattern

**The golden rule:** Choose the database that matches your ACCESS PATTERNS, not your data model.

---

```
╔═══════════════════════════════════════════════════════════════════╗
║  DATABASE LANDSCAPE                                              ║
╠═══════════════════════════════════════════════════════════════════╣
║                                                                   ║
║  Type           │ Examples          │ Best For                   ║
║  ───────────────│───────────────────│────────────────────────    ║
║  Relational     │ PostgreSQL, MySQL │ Structured data, ACID     ║
║  (SQL)          │ Oracle, SQL Server│ transactions, JOINs       ║
║                 │                   │                            ║
║  Document       │ MongoDB, CouchDB  │ Flexible schema, nested   ║
║                 │ Firestore         │ objects, rapid iteration   ║
║                 │                   │                            ║
║  Key-Value      │ Redis, DynamoDB   │ Caching, sessions, simple ║
║                 │ Memcached         │ lookups, high throughput   ║
║                 │                   │                            ║
║  Wide-Column    │ Cassandra, HBase  │ Time-series, logging,     ║
║                 │ ScyllaDB          │ massive write throughput   ║
║                 │                   │                            ║
║  Graph          │ Neo4j, Neptune    │ Relationships, social     ║
║                 │ ArangoDB          │ networks, recommendations ║
║                 │                   │                            ║
║  Search         │ Elasticsearch     │ Full-text search, logs,   ║
║                 │ OpenSearch, Solr  │ analytics                 ║
║                 │                   │                            ║
║  Time-Series    │ InfluxDB, TimescaleDB │ Metrics, IoT, stocks ║
║                 │ QuestDB           │                            ║
║                                                                   ║
╚═══════════════════════════════════════════════════════════════════╝
```

---

In [ ]:
# ============================================================
# 1. SQL vs NoSQL Decision Framework
# ============================================================

print("""
SQL vs NoSQL — THE DECISION FRAMEWORK
══════════════════════════════════════

Choose SQL when:                    Choose NoSQL when:
─────────────────                   ──────────────────
✅ ACID is critical                 ✅ Flexible/evolving schema
✅ Complex JOINs needed             ✅ Horizontal scaling required
✅ Fixed schema (known upfront)     ✅ High write throughput
✅ Strong consistency required      ✅ Denormalized data OK
✅ Financial transactions           ✅ Semi-structured data (JSON)
✅ Reporting / analytics            ✅ Low-latency lookups

Examples:
─────────
Banking app → PostgreSQL (ACID, transactions)
Social media feed → MongoDB (flexible posts) + Redis (cache)
Real-time chat → Cassandra (write-heavy, time-series)
Product catalog → MongoDB (varied attributes per product)
User sessions → Redis (fast key-value lookups)
Friend graph → Neo4j (relationship traversals)
Log analytics → Elasticsearch (full-text search + aggregation)

💡 Interview tip: "In practice, most systems use MULTIPLE databases.
   PostgreSQL for core data + Redis for cache + Elasticsearch for search.
   This is called Polyglot Persistence."
""")

In [ ]:
# ============================================================
# 2. Read Replicas & Master-Slave Pattern
# ============================================================

print("""
READ REPLICAS PATTERN
═════════════════════

Problem: Single DB can't handle high read traffic.
Solution: One master for writes, N replicas for reads.

                    ┌──── Read Replica 1 ◄── reads
  writes ──► Master ┤
              │      ├──── Read Replica 2 ◄── reads
              │      │
              │      └──── Read Replica 3 ◄── reads
              │
              └──► replication (async or sync)

Replication types:
  Synchronous:  Master waits for replica ACK → strong consistency, slower
  Asynchronous: Master doesn't wait → eventual consistency, faster
  Semi-sync:    Wait for 1 replica ACK → balance

Replication lag:
  User writes → reads from replica → doesn't see own write!
  Fix: Read-your-writes consistency
    - Read from master for X seconds after write
    - Or route user's own reads to master

When master dies:
  Promote a replica to master (failover)
  Other replicas point to new master
  → Automated in AWS RDS, GCP Cloud SQL
""")

# Quick capacity math
print("Capacity example:")
single_db_qps = 10_000
read_ratio = 0.9  # 90% reads
total_qps = 50_000
read_qps = total_qps * read_ratio
replicas_needed = read_qps / single_db_qps
print(f"  Total QPS: {total_qps:,}")
print(f"  Read QPS (90%): {read_qps:,.0f}")
print(f"  Single DB handles: {single_db_qps:,} QPS")
print(f"  Read replicas needed: {replicas_needed:.0f}")
print(f"  + 1 master for writes = {replicas_needed + 1:.0f} total DBs")

In [ ]:
# ============================================================
# 3. Indexing Strategy
# ============================================================

print("""
DATABASE INDEXING — INTERVIEW ESSENTIALS
════════════════════════════════════════

Index = data structure that speeds up reads at the cost of slower writes.

Types:
┌─────────────────┬───────────────────────┬──────────────────────┐
│ Index Type      │ How it works          │ Best for             │
├─────────────────┼───────────────────────┼──────────────────────┤
│ B-Tree / B+Tree │ Balanced tree, sorted │ Range queries, ORDER │
│ (default)       │ O(log N) lookup       │ BY, general purpose  │
│                 │                       │                      │
│ Hash            │ Hash table            │ Exact match only     │
│                 │ O(1) lookup           │ No range support     │
│                 │                       │                      │
│ GIN             │ Generalized Inverted  │ Full-text search,    │
│                 │ Index                 │ JSON, arrays         │
│                 │                       │                      │
│ GiST            │ Generalized Search    │ Geospatial, range    │
│                 │ Tree                  │ types                │
│                 │                       │                      │
│ Composite       │ Multi-column index    │ Queries on multiple  │
│                 │ (a, b, c)             │ columns together     │
└─────────────────┴───────────────────────┴──────────────────────┘

Indexing rules of thumb:
  ✅ Index columns in WHERE, JOIN, ORDER BY
  ✅ Composite index: leftmost prefix rule (a,b,c) covers (a), (a,b), (a,b,c)
  ✅ Covering index: include all SELECT columns → no table lookup
  ❌ Don't index columns with low cardinality (e.g., boolean)
  ❌ Don't index write-heavy tables excessively
  ❌ Too many indexes → slow inserts/updates

  Rule: ≤ 5 indexes per table in OLTP systems.
""")

In [ ]:
# ============================================================
# 4. Connection Pooling
# ============================================================

print("""
CONNECTION POOLING
══════════════════

Problem: Creating a DB connection is EXPENSIVE (TCP handshake, auth, SSL).
         ~10-50ms per connection creation.

Solution: Maintain a pool of reusable connections.

  App Server                     DB
  ┌──────────┐                ┌────────┐
  │ Request 1│──► Pool ─────►│        │
  │ Request 2│──► [C1] ────►│ Postgres│
  │ Request 3│──► [C2] ────►│        │
  │  (waits) │    [C3] ────►│        │
  └──────────┘    [C4] idle  └────────┘

Key parameters:
  min_connections: 5-10   (pre-warmed)
  max_connections: 20-50  (per server)
  idle_timeout:    300s   (close idle connections)
  connection_timeout: 5s  (wait for available connection)

PostgreSQL default max_connections = 100
With 10 app servers × 20 pool size = 200 connections → NEED PgBouncer!

PgBouncer: Connection pooler that sits between app and DB
  Modes:
  - Session:     1 client = 1 server conn (default, safest)
  - Transaction: conn returned after each txn (most efficient)
  - Statement:   conn returned after each query (most aggressive)

💡 Interview: "I'd use PgBouncer in transaction mode to support
   hundreds of app servers connecting to a single PostgreSQL instance."
""")

In [ ]:
# ============================================================
# 5. Database-per-Service Pattern
# ============================================================

print("""
╔════════════════════════════════════════════════════════════════╗
║  DATABASE-PER-SERVICE PATTERN (Microservices)                 ║
╠════════════════════════════════════════════════════════════════╣
║                                                                ║
║  ❌ Shared DB:                ✅ DB per Service:              ║
║                                                                ║
║  [User Svc]──┐               [User Svc]───[PostgreSQL]       ║
║  [Order Svc]──┼──[Single DB]  [Order Svc]──[PostgreSQL]      ║
║  [Notif Svc]──┘               [Notif Svc]─[MongoDB]          ║
║                                [Search Svc]─[Elasticsearch]   ║
║  Problems:                     [Cache Svc]──[Redis]           ║
║  - Tight coupling                                             ║
║  - Schema changes break       Benefits:                      ║
║    all services               - Independent scaling           ║
║  - Can't scale DB per svc     - Right tool for the job       ║
║  - Performance contention     - Independent deployment       ║
║                                                                ║
║  Cross-service data needs?                                    ║
║  → API calls (sync) or Events (async)                        ║
║  → NEVER direct DB access across services!                   ║
║                                                                ║
║  Consistency across services:                                 ║
║  → Saga pattern (choreography or orchestration)              ║
║  → Eventual consistency with events                          ║
║                                                                ║
╚════════════════════════════════════════════════════════════════╝
""")

---

## 📝 Interview Questions

| # | Question | Key Points |
|---|----------|-----------|
| 1 | SQL vs NoSQL? | SQL = ACID, JOINs, fixed schema. NoSQL = flexible, horizontal scaling, denormalized |
| 2 | When would you choose MongoDB? | Flexible schema, rapid prototyping, nested documents, no complex JOINs needed |
| 3 | How do read replicas work? | Master handles writes, replicas sync via replication log, route reads to replicas |
| 4 | What is replication lag? | Delay between master write and replica update. Fix: read-your-writes consistency |
| 5 | When to use Cassandra? | Massive write throughput, time-series, eventually consistent OK, no complex queries |
| 6 | Database per service — tradeoffs? | Independence + right tool, but cross-service queries harder, need saga for txns |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Choose DB by access patterns, not data model          │
## │  • Most systems use polyglot persistence                 │
## │  • Read replicas for read-heavy (90%+ reads)             │
## │  • Index WHERE/JOIN/ORDER columns, ≤5 per table          │
## │  • Connection pool + PgBouncer for production            │
## │  • Database per service in microservices                 │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Message Queues — Kafka, RabbitMQ & Event-Driven**